# BiLSTM-CNN Baseline ABSA — Vietnamese Student Feedback
Kiến trúc: `Embedding → BiLSTM → CNN (kernel 2,3,4) → MaxPool → FC → Output (11 aspects × 4 classes)`

Baseline tái hiện theo paper gốc để so sánh với mBERT/PhoBERT Multi-task.

- **Label**: 4 class per aspect: `none(0) / positive(1) / neutral(2) / negative(3)`
- **Loss**: `CrossEntropyLoss` — không dùng IGNORE_INDEX
- **Không dùng BERT** — dùng vocabulary tự build + word embedding 300d


In [ ]:
!pip install pyvi tensorboardX tqdm -q
#pip install underthesea tensorboardX tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 81.3 MB/s eta 0:00:00


In [ ]:
import os, sys, random, unicodedata, re
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/bilstm'


# Gộp toàn bộ 16k từ tất cả các file
all_data_files = [f'{BASE_DIR}/train{i}.csv' for i in range(1, 7)]
all_data_files += [f'{BASE_DIR}/validation.csv', f'{BASE_DIR}/test.csv']

OUTPUT_DIR = f'{BASE_DIR}/a'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Seeds để thử
SEED_LIST = [42, 0, 123, 2024, 7]



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Config

In [ ]:
import os, sys, random, unicodedata, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import Adam
from tensorboardX import SummaryWriter
from tqdm.notebook import tqdm, trange
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from collections import Counter
#from pyvi import ViTokenizer
#from underthesea import word_tokenize

# ON_KAGGLE   = os.path.exists('/kaggle/working')
# WORKING_DIR = '/kaggle/working' if ON_KAGGLE else os.getcwd()
# print('Environment:', 'Kaggle' if ON_KAGGLE else 'Local')

# # ── Data paths (Paper-matched subset: ~5,286 pairs, 7:1:2 split) ─────────
# # Subset đã được lọc để khớp phân bố aspect-sentiment của paper gốc
# train_paper      = '/kaggle/input/datasets/conanwinner1/vku-feedback-5k/train_paper.csv'
# validation_paper = '/kaggle/input/datasets/conanwinner1/vku-feedback-5k/validation_paper.csv'
# test_paper       = '/kaggle/input/datasets/conanwinner1/vku-feedback-5k/test_paper.csv'

# OUTPUT_DIR = f'{WORKING_DIR}/bilstm-cnn-acsa'
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Aspect Categories (giữ đúng thứ tự như PhoBERT notebook) ─────────────
ASPECT_CATEGORIES = [
    'ky_nang_giang_day',    # 0
    'hanh_vi',              # 1
    'de_xuat',              # 2
    'bai_tap',              # 3
    'chuong_trinh_hoc',     # 4
    'kien_thuc',            # 5
    'kinh_nghiem',          # 6
    'cung_cap_tai_lieu',    # 7
    'thiet_bi_day_hoc',     # 8
    'cham_diem',            # 9
    'noi_chung',            # 10
]
NUM_ASPECTS = len(ASPECT_CATEGORIES)
ASPECT2ID   = {a: i for i, a in enumerate(ASPECT_CATEGORIES)}

# ── Label mapping (4 class per aspect) ───────────────────────────────────
# none=0, positive=1, neutral=2, negative=3
LABEL_MAP = {'none': 0, 'positive': 1, 'neutral': 2, 'negative': 3}
ID2LABEL  = {v: k for k, v in LABEL_MAP.items()}
NUM_CLASSES = len(LABEL_MAP)

# Mapping từ sentiment string trong CSV → label mới
# CSV: negative=0, neutral=1, positive=2  →  none=0, pos=1, neu=2, neg=3
SENTIMENT_STR_TO_LABEL = {
    'positive': 1,
    'neutral':  2,
    'negative': 3,
}

# ── Model hyperparams (theo paper) ───────────────────────────────────────
EMBEDDING_DIM  = 300
HIDDEN_SIZE    = 128
NUM_FILTERS    = 128
KERNEL_SIZES   = [2, 3, 4]
FC_HIDDEN      = 128
DROPOUT        = 0.5

# ── Training ──────────────────────────────────────────────────────────────
SEED             = 42
MAX_EPOCHS       = 50
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE  = 64
LEARNING_RATE    = 1e-3
WEIGHT_DECAY     = 1e-4
MAX_SEQ_LENGTH   = 128
PATIENCE         = 10

# ── Logging ───────────────────────────────────────────────────────────────
LOG_STEPS  = 50

print(f'Aspects   : {NUM_ASPECTS}')
print(f'Classes   : {NUM_CLASSES} (none/positive/neutral/negative)')
print(f'Output    : {OUTPUT_DIR}')


Aspects   : 11
Classes   : 4 (none/positive/neutral/negative)
Output    : /content/drive/MyDrive/bilstm/a


## 2. Kiểm tra GPU

In [ ]:
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpu  = torch.cuda.device_count()


PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4


## 3. Preprocessing & Vocabulary

In [ ]:
from pyvi import ViTokenizer

def preprocess_text(text):
    text = unicodedata.normalize('NFC', str(text).strip())
    text = re.sub(r'\s+', ' ', text)          # Step 1: xóa khoảng trắng thừa
    text = re.sub(r'[^\w\s]', ' ', text)      # Step 1: xóa dấu câu, icon
    text = re.sub(r'\d+', 'num', text)        # Step 1: số → "num"
    text = ViTokenizer.tokenize(text)         # Step 2: PyVi tokenize
    return text.lower()                       # Step 3: lowercase


# Test
for s in ['Giảng viên rất nhiệt tình.', '. Thầy/cô tôn trọng sinh viên.']:
    print(f'  {s}  →  {preprocess_text(s)}')

  Giảng viên rất nhiệt tình.  →  giảng_viên rất nhiệt_tình
  . Thầy/cô tôn trọng sinh viên.  →  thầy cô tôn_trọng sinh_viên


## 4. Load Data & Build Vocabulary

In [ ]:
from collections import Counter

def load_bilstm_data(df_file):
    """
    Group theo id → mỗi sample:
      - tokens : list[str] (đã preprocess)
      - labels : list[int] length=NUM_ASPECTS, giá trị 0-3
                 0=none, 1=positive, 2=neutral, 3=negative
    """
    df = df_file
    samples = []
    skipped = 0

    for sent_id, group in df.groupby('id'):
        text   = preprocess_text(group.iloc[0]['text'])
        tokens = text.split()

        labels = [0] * NUM_ASPECTS   # mặc định none=0

        for _, row in group.iterrows():
            asp_str = str(row['aspect']).strip()
            sen_str = str(row['sentiment']).strip()

            if asp_str not in ASPECT2ID:
                skipped += 1
                continue
            if sen_str not in SENTIMENT_STR_TO_LABEL:
                skipped += 1
                continue

            asp_idx = ASPECT2ID[asp_str]
            labels[asp_idx] = SENTIMENT_STR_TO_LABEL[sen_str]

        samples.append({'tokens': tokens, 'labels': labels})

    if skipped:
        print(f'  ⚠️  Skipped {skipped} rows')
    return samples


def build_vocab(samples, min_freq=1):
    """Build word2idx từ train set."""
    counter = Counter()
    for s in samples:
        counter.update(s['tokens'])
    word2idx = {'<PAD>': 0, '<UNK>': 1}
    for word, freq in counter.items():
        if freq >= min_freq:
            word2idx[word] = len(word2idx)
    return word2idx


def print_stats(samples, split_name):
    print(f'\n  {split_name}: {len(samples)} sentences')
    aspect_counts = [0] * NUM_ASPECTS
    label_counts  = Counter()
    for s in samples:
        for i, lbl in enumerate(s['labels']):
            if lbl != 0:
                aspect_counts[i] += 1
                label_counts[ID2LABEL[lbl]] += 1
    print('  Aspect distribution (non-none):')
    max_cnt = max(aspect_counts) or 1
    for i, cat in enumerate(ASPECT_CATEGORIES):
        bar = '█' * int(20 * aspect_counts[i] / max_cnt)
        print(f'    {i:2d}. {cat:<25s} {aspect_counts[i]:4d}  {bar}')
    print('  Label distribution:', dict(label_counts))



# Các biến dev_s, test_s, word2idx sẽ được tạo ở cell sample bên dưới
print("Sẵn sàng load data.")


Sẵn sàng load data.


In [ ]:
# ── Tỉ lệ Bảng 2 paper ───────────────────────────────────────────────────
PAPER_ASPECT_COUNTS = {
    'ky_nang_giang_day': 1699,
    'hanh_vi':           1970,
    'bai_tap':            277,
    'cham_diem':           62,
    'cung_cap_tai_lieu':  155,
    'kien_thuc':          214,
    'kinh_nghiem':        120,
    'chuong_trinh_hoc':    78,
    'thiet_bi_day_hoc':    64,
    'de_xuat':            152,
    'noi_chung':          567,
}
PAPER_TOTAL_LABELS = sum(PAPER_ASPECT_COUNTS.values())  # 5358
TARGET_TOTAL       = 5010   # tổng câu paper
TARGET_TRAIN       = 3507   # 70%
TARGET_VAL         = 501    # 10%
TARGET_TEST        = 1002   # 20%


def load_all_16k(all_data_files):
    """Gộp toàn bộ 16k từ tất cả file, tạo ID duy nhất."""
    dfs = []
    for i, f in enumerate(all_data_files):
        df_tmp = pd.read_csv(f)
        df_tmp['id'] = i * 100000 + df_tmp['id']
        dfs.append(df_tmp)
    df_all = pd.concat(dfs, ignore_index=True)
    print(f"Tổng: {df_all['id'].nunique()} câu | {len(df_all)} rows")
    return df_all


def sample_5010_stratified(df_all, target=5010, seed=42):
    """
    Exact greedy sampling:
    - Xử lý aspect từ hiếm → phổ biến
    - Mỗi aspect chọn đúng số câu theo Bảng 2
    - Ưu tiên câu chưa được chọn (tránh duplicate)
    - Cuối cùng trim/fill về đúng 5010
    """
    np.random.seed(seed)

    # Precompute: aspect → list IDs, shuffle để random
    id_to_aspects = {}
    for sid, group in df_all.groupby('id'):
        id_to_aspects[sid] = set(group['aspect'].unique()) & set(ASPECT_CATEGORIES)

    aspect_to_ids = {asp: [] for asp in ASPECT_CATEGORIES}
    for sid, aspects in id_to_aspects.items():
        for asp in aspects:
            if asp in aspect_to_ids:
                aspect_to_ids[asp].append(sid)

    # Shuffle từng aspect pool
    for asp in ASPECT_CATEGORIES:
        np.random.shuffle(aspect_to_ids[asp])

    # Scale target theo tỉ lệ (target/5010 — ở đây target=5010 nên scale=1)
    scale = target / 5010
    aspect_target = {
        asp: round(PAPER_ASPECT_COUNTS[asp] * scale)
        for asp in ASPECT_CATEGORIES
    }

    # Sắp xếp aspect từ hiếm → phổ biến (theo paper count)
    sorted_aspects = sorted(ASPECT_CATEGORIES, key=lambda a: PAPER_ASPECT_COUNTS[a])

    sampled_ids   = set()
    aspect_filled = {asp: 0 for asp in ASPECT_CATEGORIES}

    # ── Pass 1: Ưu tiên câu chỉ có 1 aspect (pure sentences) ────────────
    for asp in sorted_aspects:
        need = aspect_target[asp] - aspect_filled[asp]
        if need <= 0:
            continue

        # Câu chỉ có đúng aspect này (pure)
        pure_ids = [
            sid for sid in aspect_to_ids[asp]
            if sid not in sampled_ids and id_to_aspects[sid] == {asp}
        ]
        chosen = pure_ids[:need]
        for sid in chosen:
            sampled_ids.add(sid)
            for a in id_to_aspects[sid]:
                if a in aspect_filled:
                    aspect_filled[a] += 1

    # ── Pass 2: Lấy thêm câu multi-label cho aspect còn thiếu ───────────
    for asp in sorted_aspects:
        need = aspect_target[asp] - aspect_filled[asp]
        if need <= 0:
            continue

        candidates = [
            sid for sid in aspect_to_ids[asp]
            if sid not in sampled_ids
        ]
        # Ưu tiên câu có ít aspect nhất (ít side-effect nhất)
        candidates.sort(key=lambda sid: len(id_to_aspects[sid]))

        for sid in candidates:
            if aspect_filled[asp] >= aspect_target[asp]:
                break
            sampled_ids.add(sid)
            for a in id_to_aspects[sid]:
                if a in aspect_filled:
                    aspect_filled[a] += 1

    # ── Phase 3: Fill đủ target nếu thiếu ────────────────────────────────
    if len(sampled_ids) < target:
        all_ids   = list(df_all['id'].unique())
        remaining = [sid for sid in all_ids if sid not in sampled_ids]
        np.random.shuffle(remaining)
        for sid in remaining:
            if len(sampled_ids) >= target:
                break
            sampled_ids.add(sid)

    # ── Trim nếu vượt target ──────────────────────────────────────────────
    if len(sampled_ids) > target:
        # Loại bỏ câu có aspect đang over-represented nhất
        over_aspects = {
            asp for asp in ASPECT_CATEGORIES
            if aspect_filled[asp] > aspect_target[asp] * 1.1
        }
        sampled_list = list(sampled_ids)
        np.random.shuffle(sampled_list)
        to_remove = []
        for sid in sampled_list:
            if len(sampled_ids) - len(to_remove) <= target:
                break
            aspects = id_to_aspects.get(sid, set())
            if aspects & over_aspects:
                to_remove.append(sid)
        for sid in to_remove:
            sampled_ids.discard(sid)

        # Nếu vẫn còn thừa → trim random
        if len(sampled_ids) > target:
            sampled_list = list(sampled_ids)
            np.random.shuffle(sampled_list)
            sampled_ids = set(sampled_list[:target])

    print(f"Sampled: {len(sampled_ids)} câu")
    print(f"\nAspect fill vs target:")
    print(f"{'Aspect':<25} {'Filled':>8} {'Target':>8} {'Diff':>6}")
    print("-" * 50)
    for asp in ASPECT_CATEGORIES:
        diff = aspect_filled[asp] - aspect_target[asp]
        flag = ' ⚠️' if abs(diff) > aspect_target[asp] * 0.05 else ''
        print(f"{asp:<25} {aspect_filled[asp]:>8} {aspect_target[asp]:>8} {diff:>+6}{flag}")

    return df_all[df_all['id'].isin(sampled_ids)]


def split_train_val_test(df_5010, seed=42):
    """Chia 5010 câu theo tỉ lệ 7:1:2 — cố định."""
    np.random.seed(seed)
    all_ids = df_5010['id'].unique()
    np.random.shuffle(all_ids)

    n_test  = TARGET_TEST   # 1002
    n_val   = TARGET_VAL    # 501
    # train  = 3507

    test_ids  = all_ids[:n_test]
    val_ids   = all_ids[n_test:n_test + n_val]
    train_ids = all_ids[n_test + n_val:]

    df_train = df_5010[df_5010['id'].isin(train_ids)]
    df_val   = df_5010[df_5010['id'].isin(val_ids)]
    df_test  = df_5010[df_5010['id'].isin(test_ids)]

    print(f"Train: {df_train['id'].nunique()} | Val: {df_val['id'].nunique()} | Test: {df_test['id'].nunique()}")
    return df_train, df_val, df_test


def verify_distribution(samples, label=''):
    asp_counts = {asp: 0 for asp in ASPECT_CATEGORIES}
    for s in samples:
        for i, lbl in enumerate(s['labels']):
            if lbl != 0:
                asp_counts[ASPECT_CATEGORIES[i]] += 1
    total = sum(asp_counts.values())
    print(f"\n{label}: {len(samples)} câu | {total} labels")
    print(f"{'Aspect':<25} {'Count':>6} {'Thực%':>7} {'Paper%':>7} {'Diff':>6}")
    print("-" * 57)
    for asp in ASPECT_CATEGORIES:
        cnt = asp_counts[asp]
        ratio = cnt / total * 100 if total > 0 else 0
        paper_ratio = PAPER_ASPECT_COUNTS[asp] / PAPER_TOTAL_LABELS * 100
        diff = ratio - paper_ratio
        flag = ' ⚠️' if abs(diff) > 3 else ''
        print(f"{asp:<25} {cnt:>6} {ratio:>6.1f}% {paper_ratio:>6.1f}% {diff:>+5.1f}%{flag}")


# ── Thực thi ─────────────────────────────────────────────────────────────
df_all_16k = load_all_16k(all_data_files)

# Sample 5010 câu — CỐ ĐỊNH bằng seed=42
df_5010 = sample_5010_stratified(df_all_16k, target=5010, seed=42)

# Chia train/val/test — CỐ ĐỊNH
df_train_fixed, df_val_fixed, df_test_fixed = split_train_val_test(df_5010, seed=42)

# Load thành samples
dev_s  = load_bilstm_data(df_val_fixed)
test_s = load_bilstm_data(df_test_fixed)

# Build vocab tạm
word2idx = build_vocab(dev_s + test_s, min_freq=1)
VOCAB_SIZE = len(word2idx)
print(f"\nTemp vocab: {VOCAB_SIZE}")

# Verify phân bố
verify_distribution(dev_s,  'Val')
verify_distribution(test_s, 'Test')

Tổng: 16124 câu | 23859 rows
Sampled: 5010 câu

Aspect fill vs target:
Aspect                      Filled   Target   Diff
--------------------------------------------------
ky_nang_giang_day             1699     1699     +0
hanh_vi                       1970     1970     +0
de_xuat                        152      152     +0
bai_tap                        277      277     +0
chuong_trinh_hoc                78       78     +0
kien_thuc                      214      214     +0
kinh_nghiem                    120      120     +0
cung_cap_tai_lieu              155      155     +0
thiet_bi_day_hoc                64       64     +0
cham_diem                       62       62     +0
noi_chung                      567      567     +0
Train: 3507 | Val: 501 | Test: 1002

Temp vocab: 1247

Val: 501 câu | 501 labels
Aspect                     Count   Thực%  Paper%   Diff
---------------------------------------------------------
ky_nang_giang_day            146   29.1%   31.7%  -2.6%
hanh_vi        

## 4.1. Load Pretrained Word Embedding (fastText Vietnamese)

In [ ]:
!pip install gensim
# ── Load Word2VecVN (Tiếng Việt) ──────────────────────────────────────────
# Tải model từ: https://github.com/sonvx/word2vecVN
# Upload file .bin (ví dụ: baomoi.window2.vn.model.bin) lên Kaggle dataset.

import os
from gensim.models import KeyedVectors
import numpy as np

# Đổi tên file ở đây cho đúng với file bạn upload lên Kaggle
#WORD2VEC_VN_PATH = '/kaggle/input/models/conanwinner1/w2vec/pytorch/default/1/baomoi.vn.model.bin'

WORD2VEC_VN_PATH = '/content/drive/MyDrive/bilstm/baomoi.vn.model.bin'

def load_word2vec_embeddings(word2idx, embedding_dim=300, model_path=''):
    """
    Load pre-trained Word2Vec model (gensim) từ file .bin hoặc .vec.
    Hỗ trợ xử lý từ ghép (compound words) tiếng Việt.
    """
    vocab_size = len(word2idx)
    embedding_matrix = np.random.uniform(-0.1, 0.1, (vocab_size, embedding_dim)).astype(np.float32)
    embedding_matrix[0] = 0.0   # <PAD> = zero vector

    try:
        print(f"Loading Word2Vec model from: {model_path} ...")
        if model_path.endswith('.bin'):
            # Load binary format
            w2v_model = KeyedVectors.load_word2vec_format(model_path, binary=True)
        else:
            # Load text format (.vec / .txt)
            w2v_model = KeyedVectors.load_word2vec_format(model_path, binary=False)

        # Kiểm tra xem dimension của model có khớp không
        actual_dim = w2v_model.vector_size
        if actual_dim != embedding_dim:
            print(f"⚠️ Cảnh báo: Model dimension ({actual_dim}) khác với cấu hình ({embedding_dim}).")
            print("→ Đang tự động điều chỉnh EMBEDDING_DIM...")
            embedding_matrix = np.random.uniform(-0.1, 0.1, (vocab_size, actual_dim)).astype(np.float32)
            embedding_matrix[0] = 0.0
            embedding_dim = actual_dim

        found = 0
        found_subword = 0

        for word, idx in word2idx.items():
            if word in ('<PAD>', '<UNK>'):
                continue

            # Exact match
            if word in w2v_model:
                embedding_matrix[idx] = w2v_model[word]
                found += 1
            # Xử lý từ ghép tiếng Việt (VD: "giảng_viên" -> trung bình "giảng" và "viên")
            elif '_' in word:
                subwords = word.split('_')
                sub_vecs = [w2v_model[sw] for sw in subwords if sw in w2v_model]
                if sub_vecs:
                    embedding_matrix[idx] = np.mean(sub_vecs, axis=0)
                    found_subword += 1

        total_found = found + found_subword
        coverage = total_found / (vocab_size - 2) * 100
        print(f"  Exact match: {found} words")
        print(f"  Sub-word avg match: {found_subword} words")
        print(f"  Coverage: {total_found}/{vocab_size-2} words ({coverage:.1f}%)")
        print("✓ Successfully loaded pre-trained Word2Vec embeddings")

    except Exception as e:
        print(f"⚠️ Error loading Word2Vec: {e}")
        print("⚠️ Using random init embeddings instead.")

    return np.array(embedding_matrix, dtype=np.float32)

if os.path.exists(WORD2VEC_VN_PATH):
    pretrained_embeddings = load_word2vec_embeddings(word2idx, EMBEDDING_DIM, WORD2VEC_VN_PATH)
else:
    print(f"⚠️ Không tìm thấy file {WORD2VEC_VN_PATH}")
    print("⚠️ Chuyển sang dùng random embeddings.")
    pretrained_embeddings = None


Loading Word2Vec model from: /content/drive/MyDrive/bilstm/baomoi.vn.model.bin ...
  Exact match: 1196 words
  Sub-word avg match: 24 words
  Coverage: 1220/1245 words (98.0%)
✓ Successfully loaded pre-trained Word2Vec embeddings


## 5. Dataset & DataLoader

In [ ]:
class BiLSTMDataset(Dataset):
    def __init__(self, samples, word2idx, max_len):
        self.samples  = samples
        self.word2idx = word2idx
        self.max_len  = max_len
        self.unk_idx  = word2idx.get('<UNK>', 1)
        self.pad_idx  = word2idx.get('<PAD>', 0)

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s      = self.samples[idx]
        tokens = s['tokens']
        ids    = [self.word2idx.get(t, self.unk_idx) for t in tokens]

        # Truncate
        ids = ids[:self.max_len]
        # Pad
        ids = ids + [self.pad_idx] * (self.max_len - len(ids))

        return {
            'input_ids': torch.tensor(ids,          dtype=torch.long),
            'labels':    torch.tensor(s['labels'],  dtype=torch.long),
        }


#train_dataset = BiLSTMDataset(train_s, word2idx, MAX_SEQ_LENGTH)
dev_dataset  = BiLSTMDataset(dev_s,  word2idx, MAX_SEQ_LENGTH)
test_dataset = BiLSTMDataset(test_s, word2idx, MAX_SEQ_LENGTH)

#train_loader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset),    batch_size=TRAIN_BATCH_SIZE)
dev_loader   = DataLoader(dev_dataset,   sampler=SequentialSampler(dev_dataset),  batch_size=EVAL_BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  sampler=SequentialSampler(test_dataset), batch_size=EVAL_BATCH_SIZE)

#print(f'Train: {len(train_dataset)} | Dev: {len(dev_dataset)} | Test: {len(test_dataset)}')


## 6. Model — BiLSTM-CNN

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class FocalLoss(nn.Module):
    """
    Focal Loss: giảm contribution của easy examples (none dễ predict đúng)
    → model tập trung vào hard examples (non-none).
    gamma=2 là giá trị chuẩn từ paper gốc Focal Loss.
    """
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.weight = weight
        self.gamma  = gamma

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt      = torch.exp(-ce_loss)
        focal   = (1 - pt) ** self.gamma * ce_loss
        return focal.mean()


class BiLSTMCNN_ABSA(nn.Module):
    """
    Embedding(300) → BiLSTM(128×2=256) → CNN(k=2,3,4, 128 filters each)
    → GlobalMaxPool → Concat(384) → FC(128) → Output(11×4)

    Sử dụng per-aspect weighted CrossEntropyLoss để xử lý class imbalance.
    """
    def __init__(self, vocab_size, embedding_dim=300, hidden_size=128,
                 num_aspects=11, num_classes=4, num_filters=128,
                 kernel_sizes=(2,3,4), fc_hidden=128, dropout=0.5,
                 pretrained_embeddings=None, padding_idx=0,
                 aspect_class_weights=None):
        super().__init__()
        self.num_aspects = num_aspects
        self.num_classes = num_classes

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)

        if pretrained_embeddings is not None:
            self.embedding.weight.data.copy_(
                torch.from_numpy(np.array(pretrained_embeddings, dtype=np.float32))
            )
            self.embedding.weight.data[padding_idx] = 0.0  # PAD = zero

        self.bilstm = nn.LSTM(
            input_size=embedding_dim, hidden_size=hidden_size,
            batch_first=True, bidirectional=True,

        )
        lstm_out_dim = hidden_size * 2   # 256

        self.convs = nn.ModuleList([
            nn.Conv1d(lstm_out_dim, num_filters, kernel_size=k)
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(num_filters * len(kernel_sizes), fc_hidden)
        self.output  = nn.Linear(fc_hidden, num_aspects * num_classes)

        self.loss_fns = nn.ModuleList()
        if aspect_class_weights is not None:
            for w in aspect_class_weights:
                self.loss_fns.append(nn.CrossEntropyLoss(weight=w))
        else:
            for _ in range(num_aspects):
                self.loss_fns.append(nn.CrossEntropyLoss())

    def forward(self, input_ids, labels=None):
        emb      = self.embedding(input_ids)          # (B, L, 300)
        lstm_out, _ = self.bilstm(emb)                # (B, L, 256)
        x        = lstm_out.transpose(1, 2)           # (B, 256, L)

        pooled = []
        for conv in self.convs:
            c = F.relu(conv(x))                       # (B, 128, L-k+1)
            p = F.max_pool1d(c, c.size(2)).squeeze(2) # (B, 128)
            pooled.append(p)

        x = torch.cat(pooled, dim=1)                  # (B, 384)
        x = self.dropout(x)
        x = F.relu(self.fc(x))                        # (B, 128)
        x = self.dropout(x)
        logits = self.output(x)                       # (B, 44)
        logits = logits.view(-1, self.num_aspects, self.num_classes)  # (B, 11, 4)

        loss = None
        if labels is not None:
            total_loss = 0.0
            for asp_idx in range(self.num_aspects):
                asp_logits = logits[:, asp_idx, :]    # (B, 4)
                asp_labels = labels[:, asp_idx]       # (B,)
                total_loss += self.loss_fns[asp_idx](asp_logits, asp_labels)
            loss = total_loss / self.num_aspects
        return loss, logits


def compute_aspect_class_weights(train_samples, num_aspects=11, num_classes=4, none_scale=0.3):
    """
    Tính class weights với none_scale để giảm weight của class none.
    none_scale < 1.0 → phạt nhẹ hơn khi sai none → model dám predict non-none hơn.
    """
    counts = np.zeros((num_aspects, num_classes), dtype=np.float64)
    for s in train_samples:
        for i, lbl in enumerate(s['labels']):
            counts[i, lbl] += 1.0

    weights_list = []
    for i in range(num_aspects):
        total = counts[i].sum()
        w = np.ones(num_classes, dtype=np.float64)
        present = np.where(counts[i] > 0)[0]

        if len(present) > 1:
            y_expanded = np.repeat(present, counts[i, present].astype(int))
            cw = compute_class_weight('balanced', classes=present, y=y_expanded)
            for j, cls in enumerate(present):
                w[cls] = cw[j]

        # Giảm weight của none để model không thiên về none
        w[0] = w[0] * none_scale

        # Normalize để tổng weight = num_classes
        w = w / w.mean()

        weights_list.append(torch.tensor(w, dtype=torch.float))

        asp_name = ASPECT_CATEGORIES[i] if i < len(ASPECT_CATEGORIES) else f"asp_{i}"
        none_pct = counts[i, 0] / total * 100
        print(f"  {asp_name:<25s} none={counts[i,0]:5.0f}({none_pct:5.1f}%) "
              f"pos={counts[i,1]:4.0f} neu={counts[i,2]:4.0f} neg={counts[i,3]:4.0f} "
              f"→ weights=[{w[0]:.2f}, {w[1]:.2f}, {w[2]:.2f}, {w[3]:.2f}]")

    return weights_list


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(SEED)



## 7. Training

In [ ]:
def evaluate_model(model, loader):
    model.eval()
    total_loss = 0.0
    all_gold, all_pred = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc='Evaluating', leave=False):
            input_ids = batch['input_ids'].to(device)
            labels    = batch['labels'].to(device)

            loss, logits = model(input_ids, labels)
            if n_gpu > 1: loss = loss.mean()
            total_loss += loss.item()

            pred = logits.argmax(dim=-1).cpu().numpy()   # (B, 11)
            gold = labels.cpu().numpy()                  # (B, 11)
            all_gold.append(gold)
            all_pred.append(pred)

    all_gold = np.vstack(all_gold)   # (N, 11)
    all_pred = np.vstack(all_pred)

    # Aspect Detection: pred!=0 → aspect present
    asp_gold = (all_gold != 0).astype(int)
    asp_pred = (all_pred != 0).astype(int)
    asp_f1_micro = f1_score(asp_gold, asp_pred, average='micro', zero_division=0)
    asp_f1_macro = f1_score(asp_gold, asp_pred, average='macro', zero_division=0)

    # Aspect-Sentiment: chỉ tính ở vị trí gold != 0
    # Joint evaluation: đúng khi gold!=0 VÀ pred==gold
    tp_as = fp_as = fn_as = 0
    for i in range(all_gold.shape[0]):
        for j in range(all_gold.shape[1]):
            y, p = int(all_gold[i, j]), int(all_pred[i, j])
            if y != 0 and p == y: tp_as += 1
            if y != 0 and p != y: fn_as += 1
            if p != 0 and p != y: fp_as += 1

    prec_as  = tp_as / (tp_as + fp_as) if (tp_as + fp_as) > 0 else 0
    rec_as   = tp_as / (tp_as + fn_as) if (tp_as + fn_as) > 0 else 0
    sent_f1_micro = 2 * prec_as * rec_as / (prec_as + rec_as) if (prec_as + rec_as) > 0 else 0
    sent_f1_macro = sent_f1_micro  # dùng micro cho early stopping (đúng paper)

    # Giữ lại để dùng cho classification_report nếu cần
    mask = all_gold != 0
    sent_gold = all_gold[mask].tolist()
    sent_pred = all_pred[mask].tolist()

    return {
        'loss':          total_loss / len(loader),
        'asp_f1_micro':  asp_f1_micro,
        'asp_f1_macro':  asp_f1_macro,
        'sent_f1_micro': sent_f1_micro,
        'sent_f1_macro': sent_f1_macro,
        'all_gold':      all_gold,
        'all_pred':      all_pred,
        'sent_gold':     sent_gold,
        'sent_pred':     sent_pred,
    }


In [ ]:
from copy import deepcopy

def run_one_seed(seed, df_train, dev_s, test_s):
    print(f"\n{'='*60}")
    print(f"SEED = {seed}")
    print(f"{'='*60}")
    history = {'epoch': [], 'train_loss': [], 'val_loss': [],
           'val_asp_f1_micro': [], 'val_asp_f1_macro': [],
           'val_sent_f1_micro': [], 'val_sent_f1_macro': []}
    # 1. Sample 5k train
    set_seed(seed)
    # Train đã cố định — không sample lại theo seed
    # Chỉ thay đổi seed cho model init và dataloader shuffle
    train_s_seed = load_bilstm_data(df_train)
    dev_s_seed   = dev_s
    test_s_seed  = test_s
    verify_distribution(train_s_seed, f'Train seed={seed}')
    print(f"Train sampled: {len(train_s_seed)} sentences")




    # 2. Build vocab + embedding
    word2idx_seed = build_vocab(train_s_seed, min_freq=1)
    vocab_size_seed = len(word2idx_seed)
    emb_matrix = load_word2vec_embeddings(word2idx_seed, EMBEDDING_DIM, WORD2VEC_VN_PATH)

    # 3. Dataset & Loader
    train_dataset = BiLSTMDataset(train_s_seed, word2idx_seed, MAX_SEQ_LENGTH)
    dev_dataset   = BiLSTMDataset(dev_s_seed,   word2idx_seed, MAX_SEQ_LENGTH)
    test_dataset  = BiLSTMDataset(test_s_seed,  word2idx_seed, MAX_SEQ_LENGTH)

    train_loader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset),
                              batch_size=TRAIN_BATCH_SIZE)
    dev_loader   = DataLoader(dev_dataset,   sampler=SequentialSampler(dev_dataset),
                              batch_size=EVAL_BATCH_SIZE)
    test_loader  = DataLoader(test_dataset,  sampler=SequentialSampler(test_dataset),
                              batch_size=EVAL_BATCH_SIZE)

    # 4. Model
    set_seed(seed)
    print("  Computing class weights...")
    aspect_weights_device = None

    m = BiLSTMCNN_ABSA(
        vocab_size=vocab_size_seed, embedding_dim=EMBEDDING_DIM,
        hidden_size=HIDDEN_SIZE, num_aspects=NUM_ASPECTS, num_classes=NUM_CLASSES,
        num_filters=NUM_FILTERS, kernel_sizes=KERNEL_SIZES,
        fc_hidden=FC_HIDDEN, dropout=DROPOUT,
        pretrained_embeddings=emb_matrix,
        aspect_class_weights=aspect_weights_device,
    ).to(device)

    # Freeze embedding 3 epoch đầu
    #m.embedding.weight.requires_grad = False

    optimizer = Adam(m.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3
    )

    best_val_f1  = 0.0
    patience_cnt = 0
    best_state   = None

    for epoch in range(MAX_EPOCHS):
        # Unfreeze embedding từ epoch 4
        # if epoch == 3:
        #     m.embedding.weight.requires_grad = True
        #     optimizer.add_param_group(
        #         {'params': m.embedding.parameters(),
        #          'lr': LEARNING_RATE * 0.1, 'weight_decay': 0}
        #     )
        #     print("  → Embedding unfrozen (lr=1e-4)")

        m.train()
        epoch_loss = 0.0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            labels    = batch['labels'].to(device)
            loss, _   = m(input_ids, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            optimizer.step(); optimizer.zero_grad()
            epoch_loss += loss.item()

        val_metrics = evaluate_model(m, dev_loader)
        scheduler.step(val_metrics['asp_f1_micro'])
        history['epoch'].append(epoch + 1)
        history['train_loss'].append(epoch_loss / len(train_loader))
        history['val_loss'].append(val_metrics['loss'])
        history['val_asp_f1_micro'].append(val_metrics['asp_f1_micro'])
        history['val_asp_f1_macro'].append(val_metrics['asp_f1_macro'])
        history['val_sent_f1_micro'].append(val_metrics['sent_f1_micro'])
        history['val_sent_f1_macro'].append(val_metrics['sent_f1_macro'])

        print(f"  [E{epoch+1:02d}] loss={epoch_loss/len(train_loader):.4f} "
              f"| asp_f1={val_metrics['asp_f1_micro']:.4f} "
              f"| sent_f1={val_metrics['sent_f1_micro']:.4f}")

        if val_metrics['asp_f1_micro'] > best_val_f1:
            best_val_f1 = val_metrics['asp_f1_micro']
            patience_cnt = 0
            best_state = deepcopy(m.state_dict())
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    # Load best và evaluate test
    m.load_state_dict(best_state)
    test_metrics = evaluate_model(m, test_loader)

    print(f"\n  SEED {seed} RESULT:")
    print(f"  asp_f1_micro  = {test_metrics['asp_f1_micro']:.4f}")
    print(f"  sent_f1_micro = {test_metrics['sent_f1_micro']:.4f}")

    return test_metrics, m, word2idx_seed, history, dev_s_seed, test_s_seed


# ── Chạy toàn bộ seeds ────────────────────────────────────────────────────
all_results = {}
for seed in SEED_LIST:
    metrics, trained_model, trained_w2idx, hist, dev_s_sd, test_s_sd = run_one_seed(seed, df_train_fixed, dev_s, test_s
    )
    all_results[seed] = {
        'asp_f1_micro':  metrics['asp_f1_micro'],
        'sent_f1_micro': metrics['sent_f1_micro'],
        'metrics':       metrics,
        'model':         trained_model,
        'word2idx':      trained_w2idx,
        'history':       hist,
        'dev_s':  dev_s_sd,
        'test_s': test_s_sd,
    }

# ── Tổng kết ─────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("TỔNG KẾT TẤT CẢ SEEDS")
print(f"{'='*60}")
best_seed = max(all_results, key=lambda s: all_results[s]['asp_f1_micro'])
for seed, res in all_results.items():
    marker = ' ← BEST' if seed == best_seed else ''
    print(f"  Seed {seed:4d}: asp_f1={res['asp_f1_micro']:.4f} | "
          f"sent_f1={res['sent_f1_micro']:.4f}{marker}")

# Gán model + word2idx + recreate loaders từ best seed
model    = all_results[best_seed]['model']
word2idx = all_results[best_seed]['word2idx']

# Recreate test_loader và dev_loader với word2idx của best seed
best_seed_data = all_results[best_seed]
dev_dataset_best  = BiLSTMDataset(best_seed_data['dev_s'],  word2idx, MAX_SEQ_LENGTH)
test_dataset_best = BiLSTMDataset(best_seed_data['test_s'], word2idx, MAX_SEQ_LENGTH)
dev_loader  = DataLoader(dev_dataset_best,  sampler=SequentialSampler(dev_dataset_best),
                         batch_size=EVAL_BATCH_SIZE)
test_loader = DataLoader(test_dataset_best, sampler=SequentialSampler(test_dataset_best),
                         batch_size=EVAL_BATCH_SIZE)
history = all_results[best_seed]['history']
print(f"\nBest seed: {best_seed}")
print(f"Dev loader:  {len(dev_dataset_best)} samples")
print(f"Test loader: {len(test_dataset_best)} samples")


SEED = 42

Train seed=42: 3507 câu | 3507 labels
Aspect                     Count   Thực%  Paper%   Diff
---------------------------------------------------------
ky_nang_giang_day           1146   32.7%   31.7%  +1.0%
hanh_vi                     1282   36.6%   36.8%  -0.2%
de_xuat                       93    2.7%    2.8%  -0.2%
bai_tap                      175    5.0%    5.2%  -0.2%
chuong_trinh_hoc              47    1.3%    1.5%  -0.1%
kien_thuc                    138    3.9%    4.0%  -0.1%
kinh_nghiem                   74    2.1%    2.2%  -0.1%
cung_cap_tai_lieu            105    3.0%    2.9%  +0.1%
thiet_bi_day_hoc              41    1.2%    1.2%  -0.0%
cham_diem                     36    1.0%    1.2%  -0.1%
noi_chung                    370   10.6%   10.6%  -0.0%
Train sampled: 3507 sentences
Loading Word2Vec model from: /content/drive/MyDrive/bilstm/baomoi.vn.model.bin ...
  Exact match: 1748 words
  Sub-word avg match: 50 words
  Coverage: 1798/1830 words (98.3%)
✓ Successfully

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

  [E01] loss=0.3509 | asp_f1=0.6414 | sent_f1=0.5328


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

  [E02] loss=0.1914 | asp_f1=0.6950 | sent_f1=0.6548


...

  [E20] loss=0.0226 | asp_f1=0.8330 | sent_f1=0.7766


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

  [E21] loss=0.0209 | asp_f1=0.8305 | sent_f1=0.7720
  Early stopping at epoch 21


Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]


  SEED 7 RESULT:
  asp_f1_micro  = 0.8112
  sent_f1_micro = 0.7584

TỔNG KẾT TẤT CẢ SEEDS
  Seed   42: asp_f1=0.8195 | sent_f1=0.7598
  Seed    0: asp_f1=0.8201 | sent_f1=0.7538 ← BEST
  Seed  123: asp_f1=0.8188 | sent_f1=0.7598
  Seed 2024: asp_f1=0.8080 | sent_f1=0.7401
  Seed    7: asp_f1=0.8112 | sent_f1=0.7584

Best seed: 0
Dev loader:  501 samples
Test loader: 1002 samples


## 8. Evaluation

In [ ]:
print('=== Dev ===')
dev_metrics = evaluate_model(model, dev_loader)
print(f"asp_f1_micro={dev_metrics['asp_f1_micro']:.4f} | asp_f1_macro={dev_metrics['asp_f1_macro']:.4f}")
print(f"sent_f1_micro={dev_metrics['sent_f1_micro']:.4f} | sent_f1_macro={dev_metrics['sent_f1_macro']:.4f}")

print('\n=== Test ===')
test_metrics = evaluate_model(model, test_loader)
print(f"asp_f1_micro={test_metrics['asp_f1_micro']:.4f} | asp_f1_macro={test_metrics['asp_f1_macro']:.4f}")
print(f"sent_f1_micro={test_metrics['sent_f1_micro']:.4f} | sent_f1_macro={test_metrics['sent_f1_macro']:.4f}")

print('\nPer-class Sentiment (Test, bỏ none):')
print(classification_report(
    test_metrics['sent_gold'], test_metrics['sent_pred'],
    labels=[1,2,3], target_names=['positive','neutral','negative'], zero_division=0))


=== Dev ===


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

asp_f1_micro=0.8411 | asp_f1_macro=0.7176
sent_f1_micro=0.7622 | sent_f1_macro=0.7622

=== Test ===


Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]

asp_f1_micro=0.8201 | asp_f1_macro=0.6853
sent_f1_micro=0.7538 | sent_f1_macro=0.7538

Per-class Sentiment (Test, bỏ none):
              precision    recall  f1-score   support

    positive       0.96      0.83      0.89       541
     neutral       0.55      0.44      0.49        91
    negative       0.95      0.64      0.76       370

   micro avg       0.92      0.73      0.81      1002
   macro avg       0.82      0.64      0.71      1002
weighted avg       0.92      0.73      0.81      1002



## 📊 Paper-style Evaluation (Bảng 3 & Bảng 4)

Đánh giá theo đúng format paper gốc:
- **Bảng 3**: Phát hiện khía cạnh (Aspect Detection) — binary none vs non-none
- **Bảng 4**: Phát hiện khía cạnh + cảm xúc (Aspect-Sentiment Joint) — exact match

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Paper-style Evaluation: Bảng 3 & Bảng 4
# ══════════════════════════════════════════════════════════════════════════

all_gold = test_metrics['all_gold']   # (N, 11)  — giá trị 0-3
all_pred = test_metrics['all_pred']   # (N, 11)

# ──────────────────────────────────────────────────────────────────────────
# Bảng 3: Phát hiện khía cạnh (Aspect Detection)
# Binary: none (0) vs non-none (1,2,3)
# ──────────────────────────────────────────────────────────────────────────
gold_bin = (all_gold != 0).astype(int).flatten()
pred_bin = (all_pred != 0).astype(int).flatten()

tp_ad = int(((gold_bin == 1) & (pred_bin == 1)).sum())
fp_ad = int(((gold_bin == 0) & (pred_bin == 1)).sum())
fn_ad = int(((gold_bin == 1) & (pred_bin == 0)).sum())

p_ad  = tp_ad / (tp_ad + fp_ad) * 100 if (tp_ad + fp_ad) > 0 else 0
r_ad  = tp_ad / (tp_ad + fn_ad) * 100 if (tp_ad + fn_ad) > 0 else 0
f1_ad = 2 * p_ad * r_ad / (p_ad + r_ad) if (p_ad + r_ad) > 0 else 0

print("=" * 72)
print("Bảng 3. Kết quả thí nghiệm cho bài toán phát hiện khía cạnh")
print("         trên tập kiểm tra")
print("=" * 72)
print(f"{'Phương pháp':<15s} {'Độ chính xác (%)':>18s} {'Độ phủ (%)':>14s} {'Chỉ số F1 (%)':>16s}")
print("-" * 72)
print(f"{'BiLSTM-CNN':<15s} {p_ad:>18.2f} {r_ad:>14.2f} {f1_ad:>16.2f}")
print()

# ──────────────────────────────────────────────────────────────────────────
# Bảng 4: Phát hiện khía cạnh VÀ trạng thái cảm xúc tương ứng
# (Aspect-Sentiment Joint Evaluation)
#
# TP: gold != 0 AND pred == gold  (đúng aspect + đúng sentiment)
# FP: pred != 0 AND pred != gold  (dự đoán sai: sai aspect hoặc sai sentiment)
# FN: gold != 0 AND pred != gold  (bỏ sót hoặc sai sentiment)
# ──────────────────────────────────────────────────────────────────────────
tp_as = 0
fp_as = 0
fn_as = 0

for i in range(all_gold.shape[0]):
    for j in range(all_gold.shape[1]):
        y = int(all_gold[i, j])
        p = int(all_pred[i, j])

        if y != 0 and p == y:
            tp_as += 1          # Đúng aspect + đúng sentiment
        if y != 0 and p != y:
            fn_as += 1          # Bỏ sót (hoặc sai sentiment)
        if p != 0 and p != y:
            fp_as += 1          # Dự đoán sai

p_as  = tp_as / (tp_as + fp_as) * 100 if (tp_as + fp_as) > 0 else 0
r_as  = tp_as / (tp_as + fn_as) * 100 if (tp_as + fn_as) > 0 else 0
f1_as = 2 * p_as * r_as / (p_as + r_as) if (p_as + r_as) > 0 else 0

print("=" * 72)
print("Bảng 4. Kết quả thí nghiệm cho bài toán phát hiện khía cạnh")
print("         và trạng thái cảm xúc tương ứng trên tập kiểm tra")
print("=" * 72)
print(f"{'Phương pháp':<15s} {'Độ chính xác (%)':>18s} {'Độ phủ (%)':>14s} {'Chỉ số F1 (%)':>16s}")
print("-" * 72)
print(f"{'BiLSTM-CNN':<15s} {p_as:>18.2f} {r_as:>14.2f} {f1_as:>16.2f}")
print()

# ──────────────────────────────────────────────────────────────────────────
# Chi tiết per-aspect (Bảng 3 style)
# ──────────────────────────────────────────────────────────────────────────
print("=" * 72)
print("Chi tiết phát hiện khía cạnh TỪNG aspect (Aspect Detection)")
print("=" * 72)
print(f"{'Khía cạnh':<25s} {'Precision':>10s} {'Recall':>10s} {'F1':>10s} {'Support':>10s}")
print("-" * 72)

for idx, asp_name in enumerate(ASPECT_CATEGORIES):
    g = (all_gold[:, idx] != 0).astype(int)
    p = (all_pred[:, idx] != 0).astype(int)
    tp = int(((g == 1) & (p == 1)).sum())
    fp = int(((g == 0) & (p == 1)).sum())
    fn = int(((g == 1) & (p == 0)).sum())
    support = int(g.sum())

    prec = tp / (tp + fp) * 100 if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) * 100 if (tp + fn) > 0 else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

    print(f"{asp_name:<25s} {prec:>9.2f}% {rec:>9.2f}% {f1:>9.2f}% {support:>10d}")

print("-" * 72)
print(f"{'Micro-average':<25s} {p_ad:>9.2f}% {r_ad:>9.2f}% {f1_ad:>9.2f}%")
print()

# ──────────────────────────────────────────────────────────────────────────
# Chi tiết per-aspect (Bảng 4 style)
# ──────────────────────────────────────────────────────────────────────────
print("=" * 72)
print("Chi tiết phát hiện khía cạnh + cảm xúc TỪNG aspect")
print("=" * 72)
print(f"{'Khía cạnh':<25s} {'Precision':>10s} {'Recall':>10s} {'F1':>10s} {'Support':>10s}")
print("-" * 72)

for idx, asp_name in enumerate(ASPECT_CATEGORIES):
    g_col = all_gold[:, idx]
    p_col = all_pred[:, idx]

    tp = int(((g_col != 0) & (p_col == g_col)).sum())
    fn = int(((g_col != 0) & (p_col != g_col)).sum())
    fp = int(((p_col != 0) & (p_col != g_col)).sum())
    support = int((g_col != 0).sum())

    prec = tp / (tp + fp) * 100 if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) * 100 if (tp + fn) > 0 else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

    print(f"{asp_name:<25s} {prec:>9.2f}% {rec:>9.2f}% {f1:>9.2f}% {support:>10d}")

print("-" * 72)
print(f"{'Micro-average':<25s} {p_as:>9.2f}% {r_as:>9.2f}% {f1_as:>9.2f}%")

# ──────────────────────────────────────────────────────────────────────────
# So sánh với kết quả paper gốc
# ──────────────────────────────────────────────────────────────────────────
print()
print("=" * 72)
print("So sánh với kết quả paper gốc")
print("=" * 72)
print(f"{'':25s} {'Precision':>10s} {'Recall':>10s} {'F1':>10s}")
print("-" * 72)
print(f"{'Bảng 3 - Paper':<25s} {'78.78':>10s} {'79.08':>10s} {'78.93':>10s}")
print(f"{'Bảng 3 - Ours':<25s} {p_ad:>9.2f}% {r_ad:>9.2f}% {f1_ad:>9.2f}%")
print()
print(f"{'Bảng 4 - Paper':<25s} {'73.64':>10s} {'73.93':>10s} {'73.78':>10s}")
print(f"{'Bảng 4 - Ours':<25s} {p_as:>9.2f}% {r_as:>9.2f}% {f1_as:>9.2f}%")


Bảng 3. Kết quả thí nghiệm cho bài toán phát hiện khía cạnh
         trên tập kiểm tra
Phương pháp       Độ chính xác (%)     Độ phủ (%)    Chỉ số F1 (%)
------------------------------------------------------------------------
BiLSTM-CNN                   85.33          78.94            82.01

Bảng 4. Kết quả thí nghiệm cho bài toán phát hiện khía cạnh
         và trạng thái cảm xúc tương ứng trên tập kiểm tra
Phương pháp       Độ chính xác (%)     Độ phủ (%)    Chỉ số F1 (%)
------------------------------------------------------------------------
BiLSTM-CNN                   78.43          72.55            75.38

Chi tiết phát hiện khía cạnh TỪNG aspect (Aspect Detection)
Khía cạnh                  Precision     Recall         F1    Support
------------------------------------------------------------------------
ky_nang_giang_day             87.02%     82.39%     84.64%        301
hanh_vi                       91.26%     89.78%     90.51%        372
de_xuat                       89.47